# Image Classification Model Training
**Google Colab Notebook**

This notebook will train a CNN image classifier and save the model for use in the Flask web app.

## Instructions:
1. Click **Runtime** → **Change runtime type** → Select **GPU** (T4 or better)
2. Run each cell by clicking the play button or pressing `Ctrl+Enter`
3. The trained model will download automatically at the end

---

**Time estimate:** 10-20 minutes with GPU
**Expected accuracy:** 80-85% on CIFAR-10

Output: `image_classifier.h5` (trained model)

## 1. Check GPU Availability

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

# Check if GPU is available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"[OK] GPU Available: {len(gpus)} GPU(s)")
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu}")
    print("\n[OK] You're ready for fast training!")
else:
    print("[WARN] No GPU detected. Training will be slower (CPU only).")
    print("To enable GPU: Runtime → Change runtime type → GPU")

import warnings
warnings.filterwarnings('ignore')

## 2. Install Dependencies

In [ ]:
# Install additional dependencies (mostly pre-installed in Colab)
!pip install -q scikit-learn matplotlib pillow opencv-python

print("Dependencies installed!")

## 3. Choose Dataset

You have **2 options**:
- **Option A:** Use CIFAR-10 (built-in, no upload needed)
- **Option B:** Upload your own images (structured folders)

In [ ]:
import os
import shutil
from google.colab import files, drive

DATASET_CHOICE = "cifar10"  # Change to "custom" if uploading your own

if DATASET_CHOICE == "custom":
    print("Custom Dataset Selected")
    print("\nUpload your dataset with this folder structure:")
    print("")
    print("data/train/")
    print("├── class1/")
    print("│   ├── image1.jpg")
    print("│   ├── image2.jpg")
    print("│   └── ...")
    print("├── class2/")
    print("│   ├── image1.jpg")
    print("│   └── ...")
    print("└── ...")
    print("")
    print("Ready to upload? Run the next cell.")
else:
    print("CIFAR-10 Dataset Selected")
    print("10 classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck")
    print("50,000 training images, 10,000 test images")
    print("Size: 32x32 pixels, RGB")
    print("\nDataset will download automatically in the training step")

### Upload Custom Dataset (Only if using custom option above)

In [ ]:
if DATASET_CHOICE == "custom":
    print("Upload your dataset folder (must be a ZIP file)")
    print("Structure: data/train/{class1, class2, ...}/images...")
    uploaded = files.upload()

    if uploaded:
        # Extract the uploaded zip
        !unzip -q data.zip
        print("Dataset uploaded and extracted!")

        # Verify structure
        if os.path.exists('data/train'):
            classes = [d for d in os.listdir('data/train') if os.path.isdir(os.path.join('data/train', d))]
            print(f"Found {len(classes)} classes: {classes}")
            for cls in classes:
                count = len([f for f in os.listdir(f'data/train/{cls}') if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif'))])
                print(f"  - {cls}: {count} images")
        else:
            print("[ERROR] Expected 'data/train/' folder structure")
    else:
        print("No file uploaded. Using CIFAR-10 as fallback.")
        DATASET_CHOICE = "cifar10"
else:
    print("Skipping upload - using CIFAR-10")

## 4. Training Code

This cell contains the complete CNN training pipeline. Just run it!

In [ ]:
#!/usr/bin/env python3
"""
Image Classification Model Training Script
Built for Google Colab with GPU support
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import os
import cv2
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

class ImageClassifierTrainer:
    """A class to handle training of CNN image classification model"""

    def __init__(self, data_dir='data/train', img_size=(32, 32), num_classes=10):
        self.data_dir = data_dir
        self.img_size = img_size
        self.num_classes = num_classes
        self.model = None
        self.history = None
        self.class_names = []

    def load_cifar10(self):
        """Load CIFAR-10 dataset"""
        print("Loading CIFAR-10 dataset...")
        (x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

        # Normalize
        x_train = x_train.astype('float32') / 255.0
        x_test = x_test.astype('float32') / 255.0

        y_train = y_train.reshape(-1)
        y_test = y_test.reshape(-1)

        self.class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                           'dog', 'frog', 'horse', 'ship', 'truck']

        print(f"Training samples: {len(x_train)}")
        print(f"Test samples: {len(x_test)}")
        return (x_train, y_train), (x_test, y_test)

    def load_dataset_from_folders(self):
        """Load images from folder structure"""
        print(f"Loading dataset from {self.data_dir}...")
        images = []
        labels = []

        self.class_names = sorted([d for d in os.listdir(self.data_dir)
                                 if os.path.isdir(os.path.join(self.data_dir, d))])

        print(f"Found {len(self.class_names)} classes: {self.class_names}")

        for class_idx, class_name in enumerate(self.class_names):
            class_folder = os.path.join(self.data_dir, class_name)
            print(f"  Loading '{class_name}' ({class_idx})...")

            for img_name in os.listdir(class_folder):
                img_path = os.path.join(class_folder, img_name)
                try:
                    img = cv2.imread(img_path)
                    if img is None:
                        continue
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, self.img_size)
                    images.append(img)
                    labels.append(class_idx)
                except Exception as e:
                    continue

        images = np.array(images, dtype=np.float32)
        labels = np.array(labels)
        print(f"Dataset loaded: {len(images)} images")
        return images, labels

    def prepare_data(self, use_cifar10=True):
        """Prepare data for training"""
        if use_cifar10:
            (x_train, y_train), (x_val, y_val) = self.load_cifar10()
        else:
            images, labels = self.load_dataset_from_folders()
            x_train, x_val, y_train, y_val = train_test_split(
                images, labels, test_size=0.2, random_state=42, stratify=labels
            )
            x_train = x_train / 255.0
            x_val = x_val / 255.0

        # Convert to categorical
        y_train = keras.utils.to_categorical(y_train, self.num_classes)
        y_val = keras.utils.to_categorical(y_val, self.num_classes)

        print(f"\nData shapes:")
        print(f"   X_train: {x_train.shape}")
        print(f"   X_val: {x_val.shape}")
        print(f"   y_train: {y_train.shape}")
        return (x_train, y_train), (x_val, y_val)

    def build_model(self, input_shape=(32, 32, 3)):
        """Build CNN model architecture"""
        print("Building CNN model...")

        inputs = keras.Input(shape=input_shape)

        # Conv Block 1
        x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Conv Block 2
        x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Conv Block 3
        x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D((2, 2))(x)
        x = layers.Dropout(0.25)(x)

        # Dense layers
        x = layers.Flatten()(x)
        x = layers.Dense(128, activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Dropout(0.5)(x)

        # Output
        outputs = layers.Dense(self.num_classes, activation='softmax')(x)

        self.model = keras.Model(inputs=inputs, outputs=outputs, name='image_classifier')

        self.model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        print(self.model.summary())
        return self.model

    def train(self, x_train, y_train, x_val, y_val, epochs=20, batch_size=32):
        """Train the model"""
        print("\nStarting training...")

        os.makedirs('models', exist_ok=True)

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=5,
                restore_best_weights=True,
                verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=3,
                min_lr=1e-6,
                verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                'models/best_model.h5',
                monitor='val_accuracy',
                save_best_only=True,
                mode='max',
                verbose=1
            )
        ]

        print("\nTraining progress:")
        self.history = self.model.fit(
            x_train, y_train,
            validation_data=(x_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            verbose=1
        )

        print("\nTraining completed!")
        return self.history

    def plot_training_history(self):
        """Plot training curves"""
        if self.history is None:
            print("No training history!")
            return

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Accuracy
        axes[0].plot(self.history.history['accuracy'], label='Training', linewidth=2)
        axes[0].plot(self.history.history['val_accuracy'], label='Validation', linewidth=2)
        axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Accuracy')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Loss
        axes[1].plot(self.history.history['loss'], label='Training', linewidth=2)
        axes[1].plot(self.history.history['val_loss'], label='Validation', linewidth=2)
        axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Training plot saved: training_history.png")

    def save_model(self, filepath='models/image_classifier.h5'):
        """Save trained model"""
        if self.model is None:
            print("No model to save!")
            return

        self.model.save(filepath)
        print(f"Model saved: {filepath}")
        return filepath

    def evaluate(self, x_val, y_val):
        """Evaluate model"""
        if self.model is None:
            return

        test_loss, test_acc = self.model.evaluate(x_val, y_val, verbose=0)
        print(f"\nFinal Metrics:")
        print(f"   Validation Loss: {test_loss:.4f}")
        print(f"   Validation Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")
        return test_loss, test_acc


# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == '__main__':
    print("=" * 60)
    print("IMAGE CLASSIFICATION MODEL TRAINING")
    print("=" * 60)

    # Configuration
    USE_CIFAR10 = DATASET_CHOICE == 'cifar10'
    IMG_SIZE = (32, 32)
    NUM_CLASSES = 10

    # Initialize trainer
    trainer = ImageClassifierTrainer(
        data_dir='data/train',
        img_size=IMG_SIZE,
        num_classes=NUM_CLASSES
    )

    # Prepare data
    print("\n" + "=" * 60)
    print("DATA PREPARATION")
    print("=" * 60)
    (x_train, y_train), (x_val, y_val) = trainer.prepare_data(use_cifar10=USE_CIFAR10)

    # Build model
    print("\n" + "=" * 60)
    print("MODEL ARCHITECTURE")
    print("=" * 60)
    model = trainer.build_model(input_shape=x_train.shape[1:])

    # Train
    print("\n" + "=" * 60)
    print("TRAINING")
    print("=" * 60)
    history = trainer.train(
        x_train, y_train,
        x_val, y_val,
        epochs=20,
        batch_size=64  # Larger batch for GPU
    )

    # Plot
    print("\n" + "=" * 60)
    print("VISUALIZATION")
    print("=" * 60)
    trainer.plot_training_history()

    # Evaluate
    print("\n" + "=" * 60)
    print("EVALUATION")
    print("=" * 60)
    trainer.evaluate(x_val, y_val)

    # Save
    print("\n" + "=" * 60)
    print("SAVING MODEL")
    print("=" * 60)
    model_path = trainer.save_model('models/image_classifier.h5')

    print("\n" + "=" * 60)
    print("TRAINING COMPLETE!")
    print("=" * 60)
    print(f"Model saved to: {model_path}")
    print("\nNext steps: Download the model file")
    print("   Run the next cell to download.")
    print("=" * 60)

## 5. Run Training

Click the play button on this cell to start training. Watch the progress!

In [ ]:
# This cell runs the training
# All code is already above - just execute it!

print("Ready to train! Click the play button above this cell.")
print("\nIf you already ran the training code cell, you can skip to the next section.")

## 6. Download Trained Model

After training completes, run this cell to download the `.h5` model file. Then you can use it locally in the Flask app.

In [ ]:
from google.colab import files
import os

model_path = 'models/image_classifier.h5'

if os.path.exists(model_path):
    print(f"Model found: {model_path}")
    print(f"   Size: {os.path.getsize(model_path) / (1024*1024):.1f} MB")
    print("\nDownloading model...")
    files.download(model_path)
    print("\nModel downloaded!")
    print("\nNext steps:")
    print("1. Download also: training_history.png (graphs)")
    print("2. On your local machine, put image_classifier.h5 in: image-classification-system/models/")
    print("3. Run: python app.py")
else:
    print(f"[ERROR] Model not found at {model_path}")
    print("Please run the training cell first!")

## 7. (Optional) Test Prediction in Colab

Upload a test image and see the model's prediction right here in Colab.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import cv2
from google.colab import files
import matplotlib.pyplot as plt

# Load the trained model
model = keras.models.load_model('models/image_classifier.h5')

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Upload an image to test the model")
uploaded = files.upload()

if uploaded:
    img_path = list(uploaded.keys())[0]

    # Load and preprocess
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    original = img.copy()
    img = cv2.resize(img, (32, 32))
    img = img.astype('float32') / 255.0
    img = np.expand_dims(img, axis=0)

    # Predict
    predictions = model.predict(img, verbose=0)
    top_idx = np.argmax(predictions[0])
    confidence = predictions[0][top_idx]

    # Display results
    print("\n" + "=" * 50)
    print("PREDICTION RESULT")
    print("=" * 50)
    print(f"Class: {class_names[top_idx]}")
    print(f"Confidence: {confidence*100:.2f}%")
    print("=" * 50)

    # Show top 5
    top_5 = np.argsort(predictions[0])[-5:][::-1]
    print("\nTop 5 predictions:")
    for idx in top_5:
        print(f"  {class_names[idx]}: {predictions[0][idx]*100:.2f}%")

    # Plot image
    plt.figure(figsize=(6, 6))
    plt.imshow(original)
    plt.title(f"Prediction: {class_names[top_idx]} ({confidence*100:.1f}%)", fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.show()
else:
    print("No image uploaded.")

Upload an image to test the model


## Training Complete!

You've successfully:
- Trained a CNN model on CIFAR-10 (or your custom dataset)
- Saved the model to `image_classifier.h5`
- Downloaded the model file

### Next Steps:

1. **Download these files from Colab:**
   - `models/image_classifier.h5` *(main model)*
   - `training_history.png` *(optional - accuracy graphs)*

2. **On your local computer:**
   - Place `image_classifier.h5` in this folder:
     
     `image-classification-system/models/`
   - If you don't have `models` folder, create it

3. **Install dependencies locally** (see README or SETUP_GUIDE.md)

4. **Run the Flask app:**
   ```bash
   cd "image-classification-system"
   python app.py
   ```

5. **Open browser:** http://localhost:5000

6. **Upload images** and see predictions!

---

### Custom Dataset?

If you trained on your own dataset:
- Update `class_names` in `app.py` to match your classes
- Also update `class_names` in `predict.py`

```python
class_names = ['your', 'class', 'names', 'here']
```

---

## Need Help?

- **Colab issues:** Check Runtime → Change runtime type → GPU
- **Low accuracy:** Train for more epochs (try 30-50)
- **Custom dataset errors:** Ensure folder structure is correct

**Happy classifying!**